# ANOVA And Kruskal-Wallis

This notebook explains multi-group tests used for event intensity:

- no-event weeks
- low-intensity event weeks
- high-intensity event weeks

## Formulas

ANOVA compares between-group variance to within-group variance:

$$
F = \frac{MS_{between}}{MS_{within}}
$$

Kruskal-Wallis is a rank-based alternative:

$$
H = \frac{12}{N(N+1)}\sum_i n_i(\bar{R}_i - \bar{R})^2
$$

## Coding Agent Prompt

```text
Given no-event, low-intensity, and high-intensity groups, choose ANOVA or Kruskal-Wallis. Explain assumptions, group-size power concerns, and what a significant omnibus result does and does not tell us.
```

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
from scipy import stats

ROOT = next(path for path in [Path.cwd(), *Path.cwd().parents] if (path / "data" / "processed").exists())
DATA = ROOT / "data" / "processed" / "us"

In [ ]:
panel = pd.read_csv(DATA / "modeling" / "county_event_panel.csv")
positive = panel.loc[panel["event_intensity"] > 0, "event_intensity"]
threshold = positive.median()

panel["intensity_group"] = "No event"
panel.loc[(panel["event_intensity"] > 0) & (panel["event_intensity"] <= threshold), "intensity_group"] = "Low intensity"
panel.loc[panel["event_intensity"] > threshold, "intensity_group"] = "High intensity"

groups = [
    panel.loc[panel["intensity_group"].eq(label), "merchants_all"].dropna()
    for label in ["No event", "Low intensity", "High intensity"]
]
pd.Series({label: len(values) for label, values in zip(["No event", "Low intensity", "High intensity"], groups)})

In [ ]:
anova = stats.f_oneway(*groups)
kruskal = stats.kruskal(*groups)

pd.DataFrame([
    {"test": "ANOVA", "statistic": anova.statistic, "p_value": anova.pvalue},
    {"test": "Kruskal-Wallis", "statistic": kruskal.statistic, "p_value": kruskal.pvalue},
])

## Assumptions

ANOVA:

- independent observations
- approximately normal residuals within groups
- similar variance across groups
- tests whether at least one group mean differs

Kruskal-Wallis:

- independent observations
- ordinal or continuous outcome
- does not require normality
- tests whether group distributions differ

## Statistical Power Note

Multi-group tests need enough observations in each group. If one event-intensity group is rare, the test may have weak power even when the overall dataset is large.

## Interpretation

A small p-value says at least one group differs. It does not tell you which pair differs or prove a monotonic dose-response relationship.